# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 43  
**Kaggle challenge:** `Deep learning` (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "Choco Hunters"  

**Author 1 (sciper):** Ewa Miazga (367059)  
**Author 2 (sciper):** Sameh Lahouar (300454)   
**Author 3 (sciper):** Nour Guermazi (314474) 

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

## 00. Imports

In [4]:
from loader import CustomIAPRDataloader, TrainDataset, ImageOnlyDataset
from models.cnn import SimpleCNN
from helper import get_device, compute_mean_std
from trainer import Trainer
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn as nn


device = get_device()
print(f"Using device: {device}")

Using device: mps


## 00.1 Preprocess dataset with coco


In [10]:
import os
import json
import cv2
import shutil
import pandas as pd
from tqdm import tqdm
import random
import re

# split dataset_coco/train into train and val

def split_dataset(source, dest, train_ratio=0.8):
    # Load the data from the file
    data = json.load(open(os.path.join(source, "annotations.json")))

    # Map image IDs to filenames
    image_id_to_filename = {image["id"]: image["file_name"] for image in data["images"]}
    image_ids = list(image_id_to_filename.keys())

    # Shuffle and split
    random.shuffle(image_ids)
    split_index = int(len(image_ids) * train_ratio)
    train_ids = image_ids[:split_index]
    val_ids = image_ids[split_index:]

    # Create directories
    os.makedirs(dest, exist_ok=True)
    os.makedirs(os.path.join(dest, "train"), exist_ok=True)
    os.makedirs(os.path.join(dest, "val"), exist_ok=True)

    # Copy train images
    for image_id in train_ids:
        filename = image_id_to_filename[image_id]
        shutil.copy(os.path.join(source, filename), os.path.join(dest, "train", filename))

    # Copy val images
    for image_id in val_ids:
        filename = image_id_to_filename[image_id]
        shutil.copy(os.path.join(source, filename), os.path.join(dest, "val", filename))

def split_coco_annotations(source_json, dest_dir, train_ratio=0.8):
    # Load full annotations
    with open(source_json, 'r') as f:
        data = json.load(f)

    # Split images
    image_id_to_image = {img["id"]: img for img in data["images"]}
    image_ids = list(image_id_to_image.keys())
    random.shuffle(image_ids)
    split_index = int(len(image_ids) * train_ratio)

    train_ids = set(image_ids[:split_index])
    val_ids = set(image_ids[split_index:])

    # Split images
    train_images = [img for img in data["images"] if img["id"] in train_ids]
    val_images = [img for img in data["images"] if img["id"] in val_ids]

    # Split annotations
    train_annotations = [ann for ann in data["annotations"] if ann["image_id"] in train_ids]
    val_annotations = [ann for ann in data["annotations"] if ann["image_id"] in val_ids]

    # Categories stay the same
    categories = data["categories"]

    # Write new JSON files
    os.makedirs(os.path.join(dest_dir, "train"), exist_ok=True)
    os.makedirs(os.path.join(dest_dir, "val"), exist_ok=True)

    with open(os.path.join(dest_dir, "train", "annotations.json"), 'w') as f:
        json.dump({
            "images": train_images,
            "annotations": train_annotations,
            "categories": categories
        }, f)

    with open(os.path.join(dest_dir, "val", "annotations.json"), 'w') as f:
        json.dump({
            "images": val_images,
            "annotations": val_annotations,
            "categories": categories
        }, f)

    print(f"✅ Annotations split and saved to {dest_dir}/train and {dest_dir}/val")


#split_dataset("dataset_coco/train", "dataset_split_coco")
#split_coco_annotations("dataset_coco/train/annotations.json", "dataset_split_coco")

import os
import json
from pathlib import Path

def clean_and_uniquify_filenames(folder_path, prefix="img"):
    annotations_path = os.path.join(folder_path, "annotations.json")

    # Load annotations
    with open(annotations_path, 'r') as f:
        data = json.load(f)

    # Mapping from old filename → new filename
    filename_map = {}
    used_filenames = set()

    # Step 1: Generate unique new filenames
    for i, image in enumerate(data["images"]):
        old_name = image["file_name"]
        ext = Path(old_name).suffix.lower() or ".JPG"
        new_name = f"{prefix}_{str(i).zfill(5)}{ext}"

        # Ensure uniqueness
        while new_name in used_filenames:
            i += 1
            new_name = f"{prefix}_{str(i).zfill(5)}{ext}"

        used_filenames.add(new_name)
        filename_map[old_name] = new_name
        image["file_name"] = new_name  # Update in annotations

    # Step 2: Rename files on disk
    for old_name, new_name in filename_map.items():
        old_path = os.path.join(folder_path, old_name)
        new_path = os.path.join(folder_path, new_name)
        if os.path.exists(old_path):
            os.rename(old_path, new_path)
        else:
            print(f"[WARNING] Missing file: {old_path}")

    # Step 3: Save updated annotations
    with open(annotations_path, 'w') as f:
        json.dump(data, f)

    print(f"✅ Renamed {len(filename_map)} files with unique names and updated annotations.")

clean_and_uniquify_filenames("dataset_split_coco/train")
clean_and_uniquify_filenames("dataset_split_coco/val")


def extract_patch(image_path, bbox, size=1400):
    # Read the image
    image = cv2.imread(image_path)

    # Extract the bounding box coordinates
    x, y, w, h = bbox

    # Calculate the center of the bounding box
    center_x = int(x + w / 2)
    center_y = int(y + h / 2)

    # Calculate the top-left and bottom-right coordinates of the patch
    x1 = max(0, center_x - size // 2)
    y1 = max(0, center_y - size // 2)
    x2 = min(image.shape[1], center_x + size // 2)
    y2 = min(image.shape[0], center_y + size // 2)

    # Extract the patch
    patch = image[y1:y2, x1:x2]

    return patch

def patches_from_coco(source,dest):
    if not os.path.exists(os.path.join(dest, "patches")):
        os.makedirs(os.path.join(dest, "patches"))
    else:
        shutil.rmtree(os.path.join(dest, "patches"))
        os.makedirs(os.path.join(dest, "patches"))

    # Load the data from the file
    data = json.load(open(os.path.join(source, "annotations.json")))

    id_to_label = { e["id"]:e["name"] for e in data["categories"]}
    id_to_label

    id_to_images = { e["id"]:e["file_name"] for e in data["images"]}

    annotations = data["annotations"]


    df_labels = pd.DataFrame(columns=["name","label", "image", "bbox"])
    for i,annotation in tqdm(list(enumerate(annotations))):
        image_id = annotation["image_id"]
        label_id = annotation["category_id"]
        bbox = annotation["bbox"] # "bbox": [x,y,width,height]
        label = id_to_label[label_id]
        image = id_to_images[image_id]
        image_name = image.split(".")[0].replace("_", ".")
        filepath = os.path.join(source, image)
        patch = extract_patch(filepath, bbox, size=1400)
        idx = str(i).zfill(3)
        cv2.imwrite(os.path.join(dest, "patches", f"{idx}.jpg"), patch)
        df_labels.loc[i] = [idx,label, image, bbox]
    df_labels.to_csv(os.path.join(dest, "patches", "labels.csv"), index=False)

print("Processing train")
patches_from_coco("dataset_split_coco/train","processed_data/train")
print("Processing val")
patches_from_coco("dataset_split_coco/val","processed_data/val")

def patches_to_ImageFolder(src, dest):
    df = pd.read_csv(os.path.join(src,"labels.csv"))
    # empty the dest folder
    if os.path.exists(dest):
        shutil.rmtree(dest)
    # create the folder
    os.makedirs(dest, exist_ok=True)

    df["label"].unique()

    for label in tqdm(df["label"].unique()):
        os.makedirs(dest + label, exist_ok=True)
        for i, row in df[df["label"]==label].iterrows():
            idx = str(row["name"]).zfill(3)
            shutil.copy(src + idx + ".jpg", dest + label + "/" + idx + ".jpg")
print("Processing train")
patches_to_ImageFolder("processed_data/train/patches/", "processed_data/train/folder_dataset/")
print("Processing val")
patches_to_ImageFolder("processed_data/val/patches/", "processed_data/val/folder_dataset/")

✅ Renamed 90 files with unique names and updated annotations.
[WARNING] Missing file: dataset_split_coco/val/L1000854.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000928.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000875.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000788.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000964.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000817.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000896.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000756.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000885.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000872.JPG
[WARNING] Missing file: dataset_split_coco/val/L1010008.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000946.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000951.JPG
[WARNING] Missing file: dataset_split_coco/val/L1010035.JPG
[WARNING] Missing file: dataset_split_coco/val/L1000828.JPG
[WARNING] Missing file: dataset_split_

  0%|          | 0/3182 [00:00<?, ?it/s]


AttributeError: 'NoneType' object has no attribute 'shape'

## 01. Load the dataset

In [2]:
import pandas as pd
# --- Transform: resize and convert to tensor only ---
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# --- Dataset & Loader ---
csv_path = "dataset_project_iapr2025/train.csv"
image_dir = "dataset_project_iapr2025/train"
df = pd.read_csv(csv_path)
dataset = ImageOnlyDataset(image_dir=image_dir, dataframe=df, transform=transform)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=0)

mean, std = compute_mean_std(loader)
print(f"\nDataset mean: {mean}")
print(f"Dataset std: {std}")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:16<00:00,  8.32s/it]


Dataset mean: tensor([0.0153, 0.0147, 0.0145])
Dataset std: tensor([0.0038, 0.0038, 0.0043])


In [3]:
## Default dataset

transform = transforms.Compose([
    transforms.Resize((200, 300)),  # You can choose a suitable resolution
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

DATASET_DIR = "dataset_project_iapr2025"
loader = CustomIAPRDataloader(base_dir=DATASET_DIR, transform=transform)

full_train_ds = loader.train_dataset 
test_ds = loader.test_dataset
ref_ds = loader.reference_dataset
class_number = loader.class_number
class_names = loader.class_names

train_size = int(0.8 * len(full_train_ds))
val_size = len(full_train_ds) - train_size

train_ds, val_ds = random_split(full_train_ds, [train_size, val_size])

# Display information about the datasets
print(f"Train dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(val_ds)}")
print(f"Test dataset size: {len(test_ds)}")
print(f"Reference dataset size: {len(ref_ds)}")

# Create DataLoader for training and testing datasets
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

# Example of how to use the DataLoader
for images, labels in train_loader:

    print(f"Batch size: {images.size(0)}")
    print(f"Image shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    break  # Remove this to iterate through the entire dataset

# display first image from the train dataset
import matplotlib.pyplot as plt
import numpy as np
def imshow(img):
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# imshow(train_ds[0][0])

Train dataset size: 72
Validation dataset size: 18
Test dataset size: 180
Reference dataset size: 13
Batch size: 8
Image shape: torch.Size([8, 3, 200, 300])
Labels shape: torch.Size([8, 13])


In [4]:
## 02. Data Augmentation

In [5]:
from helper import unzip_to

unzip_to("../project/data_project.zip",  "../project/data/")
unzip_to("../project/data_projectv2.zip",  "../project/data/")

## 02. Create CNN model
Train, Evaluate and Predict

In [6]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = SimpleCNN(input_shape=3, hidden_units=64, image_height=200, image_width=300, output_shape=class_number)
loss_fn = nn.MSELoss()
#optimizer = torch.optim.SGD(params=model.parameters(), lr=0.05)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)

trainer = Trainer(model=model,
                      loss_fn=loss_fn,
                      optimizer=optimizer,
                      scheduler=scheduler,
                      num_epochs=5,
                      train_loader=train_loader,
                      val_loader=val_loader,
                      test_loader=test_loader,
                      device=device)

trainer.train()
#predictions = trainer.predict()
#print(predictions)
#trainer.save_model()
avg_loss, accuracy = trainer.evaluate()
#print(f"1 predition: {predictions[0][0]}")
print(f"Ground truth: {val_ds[0][1]}")
print(f"Average Loss: {avg_loss}, Accuracy: {accuracy}%")

/Users/ewamiazga/miniconda3/envs/iapr_project/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/5], Loss: 8936360.6419
predicted for first sample tensor([0., 0., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0.], device='mps:0')
Epoch [2/5], Loss: 2222.2679
predicted for first sample tensor([1., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 0.], device='mps:0')
Epoch [3/5], Loss: 0.8856
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
Epoch [4/5], Loss: 0.8824
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
Epoch [5/5], Loss: 0.8842
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
predicted for first sample tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], device='mps:0')
Ground truth: tensor([0., 0., 0., 0., 0., 3., 3., 0., 0., 0., 2., 2., 2.])
Average Loss: 1.1348543167114258, Accuracy: 0.0%


In [7]:
predictions = trainer.predict()
print(f"Predictions: {predictions[0]}")

print(f"Model summary: {model.print_model_summary()}")

Predictions: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0